# 02 - Comparative experiments (local)

Greedy sequential optimization:
- Study A: 3 embedders (baseline LLM + strict prompt)
- Study B: 3 LLMs (best embedder + strict prompt)
- Study C: 3 prompt templates (best embedder + best LLM)

In [ ]:
import sys, gc, torch
sys.path.insert(0, '..')

from src.embedding import Embedder
from src.vectorstore import VectorStore
from src.generation import LLMGenerator
from src.rag_pipeline import RAGPipeline
from evaluation.evaluate import (
    load_dataset, retrieval_recall, keyword_score, measure_latency
)
import pandas as pd

dataset = load_dataset('../evaluation/qa_dataset.json')
print('nb questions:', len(dataset))

In [ ]:
# helper to release memory between LLMs
def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        torch.mps.empty_cache()

## Study A - embedders

In [ ]:
CONFIGS_EMB = [
    ('minilm', 'sentence-transformers/all-MiniLM-L6-v2', 'collection_minilm'),
    ('e5_base', 'intfloat/multilingual-e5-base', 'collection_e5_base'),
    ('solon', 'OrdalieTech/Solon-embeddings-base-0.1', 'collection_solon_base'),
]

print('loading baseline llm (qwen 1.5b)...')
llm = LLMGenerator('Qwen/Qwen2.5-1.5B-Instruct')

results_A = []
for name, model, coll in CONFIGS_EMB:
    print(f'\n--- {name} ---')
    emb = Embedder(model)
    vs = VectorStore('../data/chroma_db', coll)
    pipe = RAGPipeline(emb, vs, llm, 'strict')
    r = retrieval_recall(pipe, dataset, k=4)
    k_score, _ = keyword_score(pipe, dataset, k=4)
    lat = measure_latency(pipe, dataset, k=4)
    results_A.append({
        'embedder': name,
        'recall@4': round(r, 3),
        'keyword': round(k_score, 3),
        'gen_ms': round(lat['generation']['median'], 0),
    })
    print(results_A[-1])

df_A = pd.DataFrame(results_A)
print(df_A)
df_A.to_csv('../evaluation/results_A_embedders.csv', index=False)

## Study B - LLMs (all running locally on mac m3 pro via MPS)

In [ ]:
best_row = df_A.sort_values('recall@4', ascending=False).iloc[0]
best_emb_name = best_row['embedder']
name_to_cfg = {n: (m, c) for (n, m, c) in CONFIGS_EMB}
best_model, best_coll = name_to_cfg[best_emb_name]
print('best embedder:', best_emb_name)

emb = Embedder(best_model)
vs = VectorStore('../data/chroma_db', best_coll)

CONFIGS_LLM = [
    ('tinyllama_1b', 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'),
    ('qwen_1.5b', 'Qwen/Qwen2.5-1.5B-Instruct'),
    ('qwen_3b', 'Qwen/Qwen2.5-3B-Instruct'),
]

results_B = []
for name, model in CONFIGS_LLM:
    print(f'\n--- {name} ---')
    try:
        del llm
    except NameError:
        pass
    free_memory()
    try:
        llm = LLMGenerator(model)
    except Exception as e:
        print(f'skip {name}: {e}')
        continue
    pipe = RAGPipeline(emb, vs, llm, 'strict')
    k_score, _ = keyword_score(pipe, dataset, k=4)
    lat = measure_latency(pipe, dataset, k=4)
    results_B.append({
        'llm': name,
        'keyword': round(k_score, 3),
        'gen_ms': round(lat['generation']['median'], 0),
    })
    print(results_B[-1])

df_B = pd.DataFrame(results_B)
print(df_B)
df_B.to_csv('../evaluation/results_B_llms.csv', index=False)

## Study C - prompts

In [ ]:
best_llm_row = df_B.sort_values('keyword', ascending=False).iloc[0]
best_llm_name = best_llm_row['llm']
name_to_llm = {n: m for (n, m) in CONFIGS_LLM}
best_llm_model = name_to_llm[best_llm_name]
print('best llm:', best_llm_name)

try:
    del llm
except NameError:
    pass
free_memory()
llm = LLMGenerator(best_llm_model)

results_C = []
examples_c = []
for prompt_name in ['strict', 'citation', 'structured']:
    print(f'\n--- prompt {prompt_name} ---')
    pipe = RAGPipeline(emb, vs, llm, prompt_name)
    k_score, details = keyword_score(pipe, dataset, k=4)
    results_C.append({'prompt': prompt_name, 'keyword': round(k_score, 3)})
    print(results_C[-1])
    for d in details[:2]:
        examples_c.append({'prompt': prompt_name, **d})

df_C = pd.DataFrame(results_C)
print(df_C)
df_C.to_csv('../evaluation/results_C_prompts.csv', index=False)
pd.DataFrame(examples_c).to_csv('../evaluation/examples_qualitative.csv', index=False)

## Summary - best config

In [ ]:
best_prompt = df_C.sort_values('keyword', ascending=False).iloc[0]['prompt']
print('retained config:')
print(f'  embedder: {best_emb_name}')
print(f'  llm: {best_llm_name}')
print(f'  prompt: {best_prompt}')
print('\nupdate app.py with these values before the demo.')